# fRG RHS optimization test notebook

This notebook does three things on a **small toy patch model**:

1. checks that the optimized kernel/flow reproduce the old kernel/flow on simple quantities,
2. profiles the RHS **by channel and by step component**, instead of timing one giant function,
3. runs a short manual flow loop and records per-step timing.

It uses:

- `frg_kernel.py` / `frg_flow.py` as the reference,
- `frg_kernel_optimized.py` / `frg_flow_optimized.py` as the optimized versions.


In [1]:
import sys, time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, "/mnt/data")

import frg_kernel as k_ref
import frg_flow as f_ref
import frg_kernel_optimized as k_opt
import frg_flow_optimized as f_opt

np.set_printoptions(precision=6, suppress=True)


## Build a minimal toy patch model

In [2]:
@dataclass
class ToyPatch:
    k_cart: np.ndarray
    energy: float

class ToyPatchSet:
    def __init__(self, ks, energies, b1=(2*np.pi, 0.0), b2=(0.0, 2*np.pi)):
        self.patches = [ToyPatch(np.asarray(k, dtype=float), float(e)) for k, e in zip(ks, energies)]
        self.Npatch = len(self.patches)
        self.b1 = np.asarray(b1, dtype=float)
        self.b2 = np.asarray(b2, dtype=float)

# 4-patch toy grid
ks = [
    (0.20, 0.10),
    (1.10, 0.20),
    (0.30, 1.00),
    (1.20, 1.10),
]
energies_up = [-0.30, -0.10, 0.10, 0.35]
energies_dn = [-0.28, -0.08, 0.12, 0.33]

patchsets = {
    "up": ToyPatchSet(ks, energies_up),
    "dn": ToyPatchSet(ks, energies_dn),
}

def toy_bare_gamma(p1, s1, p2, s2, p3, s3, p4, s4):
    # small, deterministic, spin-dependent but cheap toy vertex
    spin_factor = 1.0 if (s1 == s3 and s2 == s4) else 0.35
    patch_factor = (
        1.0
        + 0.03 * (p1 + 1)
        - 0.02 * (p2 + 1)
        + 0.04 * (p3 + 1)
        - 0.01 * (p4 + 1)
    )
    exchange = 0.02 if (s1 == s2 == s3 == s4) else -0.015
    return complex(spin_factor * patch_factor + exchange)

pp_Qs = [np.array([0.0, 0.0]), np.array([0.9, 0.0])]
ph_Qs = [np.array([0.0, 0.0]), np.array([0.9, 0.0])]
phc_Qs = [np.array([0.0, 0.0]), np.array([0.9, 0.0])]

print("toy patch count =", patchsets["up"].Npatch)


toy patch count = 4


## Create reference and optimized solvers

In [3]:
solver_ref = f_ref.FRGFlowSolver(
    patchsets=patchsets,
    bare_gamma=toy_bare_gamma,
    pp_Qs=pp_Qs,
    ph_Qs=ph_Qs,
    phc_Qs=phc_Qs,
    T_start=1.0,
    T_stop=0.7,
    n_steps=4,
    nfreq=12,
    diagnose_every=999,
)

solver_opt = f_opt.FRGFlowSolver(
    patchsets=patchsets,
    bare_gamma=toy_bare_gamma,
    pp_Qs=pp_Qs,
    ph_Qs=ph_Qs,
    phc_Qs=phc_Qs,
    T_start=1.0,
    T_stop=0.7,
    n_steps=4,
    nfreq=12,
    diagnose_every=999,
)

print("spin blocks =", solver_ref.spin_blocks)
print("temperature path =", solver_ref.temperature_path)


spin blocks = [('up', 'up', 'up', 'up'), ('dn', 'dn', 'dn', 'dn'), ('up', 'dn', 'up', 'dn'), ('up', 'dn', 'dn', 'up'), ('dn', 'up', 'up', 'dn'), ('dn', 'up', 'dn', 'up')]
temperature path = [1.       0.887904 0.788374 0.7     ]


## 1. Scalar vertex access consistency

In [4]:
gamma_ref = solver_ref.gamma_accessor()
gamma_opt = solver_opt.current_gamma_accessor()

test_points = [
    (0, "up", 1, "dn", 2, "up", 3, "dn"),
    (1, "up", 2, "up", 0, "up", 3, "up"),
    (2, "dn", 0, "up", 1, "dn", 3, "up"),
]

rows = []
for args in test_points:
    v_ref = gamma_ref(*args)
    v_opt = gamma_opt(*args)
    rows.append({
        "args": str(args),
        "ref": v_ref,
        "opt": v_opt,
        "abs_diff": abs(v_ref - v_opt),
    })

pd.DataFrame(rows)


,args,ref,opt,abs_diff
0,"(0, 'up', 1, 'dn', 2, 'up', 3, 'dn')",1.055+0.000j,1.055+0.000j,0.0
1,"(1, 'up', 2, 'up', 0, 'up', 3, 'up')",1.020+0.000j,1.020+0.000j,0.0
2,"(2, 'dn', 0, 'up', 1, 'dn', 3, 'up')",1.095+0.000j,1.095+0.000j,0.0


## 2. Kernel consistency on a tiny case

In [5]:
cfg_ref = solver_ref._flow_config(float(solver_ref.temperature_path[0]))
cfg_opt = solver_opt._flow_config(float(solver_opt.temperature_path[0]))
Q0 = np.array([0.0, 0.0])

ker_pp_ref = k_ref.compute_pp_kernel(
    gamma_ref,
    patchsets,
    Q0,
    incoming_spins=("up", "dn"),
    outgoing_spins=("up", "dn"),
    config=cfg_ref,
    allowed_spin_blocks=solver_ref.allowed_spin_blocks,
)
ker_pp_opt = k_opt.compute_pp_kernel_fast(
    gamma_opt,
    patchsets,
    Q0,
    incoming_spins=("up", "dn"),
    outgoing_spins=("up", "dn"),
    config=cfg_opt,
    allowed_spin_blocks=solver_opt.allowed_spin_blocks,
)

ker_ph_ref = k_ref.compute_ph_kernel(
    gamma_ref,
    patchsets,
    Q0,
    incoming_spins=("up", "up"),
    outgoing_spins=("up", "up"),
    config=cfg_ref,
    allowed_spin_blocks=solver_ref.allowed_spin_blocks,
)
ker_ph_opt = k_opt.compute_ph_kernel_fast(
    gamma_opt,
    patchsets,
    Q0,
    incoming_spins=("up", "up"),
    outgoing_spins=("up", "up"),
    config=cfg_opt,
    allowed_spin_blocks=solver_opt.allowed_spin_blocks,
)

ker_phc_ref = k_ref.compute_phc_kernel(
    gamma_ref,
    patchsets,
    Q0,
    incoming_spins=("up", "up"),
    outgoing_spins=("up", "up"),
    config=cfg_ref,
    allowed_spin_blocks=solver_ref.allowed_spin_blocks,
)
ker_phc_opt = k_opt.compute_phc_kernel_fast(
    gamma_opt,
    patchsets,
    Q0,
    incoming_spins=("up", "up"),
    outgoing_spins=("up", "up"),
    config=cfg_opt,
    allowed_spin_blocks=solver_opt.allowed_spin_blocks,
)

pd.DataFrame([
    {"kernel": "pp", "max_abs_diff": np.max(np.abs(ker_pp_ref.matrix - ker_pp_opt.matrix))},
    {"kernel": "ph", "max_abs_diff": np.max(np.abs(ker_ph_ref.matrix - ker_ph_opt.matrix))},
    {"kernel": "phc", "max_abs_diff": np.max(np.abs(ker_phc_ref.matrix - ker_phc_opt.matrix))},
])


,kernel,max_abs_diff
0,pp,2.220446e-16
1,ph,8.881784e-16
2,phc,8.881784e-16


## 3. Full RHS consistency on the small toy model

In [6]:
T0 = float(solver_ref.temperature_path[0])

t = time.perf_counter()
rhs_ref = solver_ref.compute_channel_rhs(T0)
t_ref = time.perf_counter() - t

t = time.perf_counter()
rhs_opt = solver_opt.compute_channel_rhs(T0)
t_opt = time.perf_counter() - t

def max_store_diff(a, b):
    keys = sorted(set(a.keys()) & set(b.keys()))
    vals = [np.max(np.abs(a[k] - b[k])) for k in keys]
    return float(max(vals)) if vals else 0.0

pd.DataFrame([
    {"store": "rhs_pp", "max_abs_diff": max_store_diff(rhs_ref[0], rhs_opt[0])},
    {"store": "rhs_phd", "max_abs_diff": max_store_diff(rhs_ref[1], rhs_opt[1])},
    {"store": "rhs_phc", "max_abs_diff": max_store_diff(rhs_ref[2], rhs_opt[2])},
    {"store": "rhs_total_time_ref", "max_abs_diff": t_ref},
    {"store": "rhs_total_time_opt", "max_abs_diff": t_opt},
    {"store": "speedup_ref_over_opt", "max_abs_diff": t_ref / t_opt if t_opt > 0 else np.inf},
])


,store,max_abs_diff
0,rhs_pp,4.440892e-16
1,rhs_phd,8.881784e-16
2,rhs_phc,1.776357e-15
3,rhs_total_time_ref,2.946233e+00
4,rhs_total_time_opt,4.903901e-01
5,speedup_ref_over_opt,6.007937e+00


## 4. Manual RHS timing, split by channel
This avoids hiding everything inside one giant function.

In [7]:
def time_rhs_by_channel_ref(solver, T):
    gamma = solver.gamma_accessor()
    cfg = solver._flow_config(T)

    timings = {}

    t = time.perf_counter()
    rhs_pp = solver._empty_channel_store(solver.pp_grid)
    for iq, Q in enumerate(solver.pp_grid.q_list):
        for s1, s2, s3, s4 in solver.spin_blocks:
            ker = k_ref.compute_pp_kernel(
                gamma, solver.patchsets, Q,
                incoming_spins=(s1, s2),
                outgoing_spins=(s3, s4),
                config=cfg,
                allowed_spin_blocks=solver.allowed_spin_blocks,
            )
            rhs_pp[(s1, s2, s3, s4, iq)] = np.asarray(ker.matrix, dtype=complex)
    timings["pp_time"] = time.perf_counter() - t

    t = time.perf_counter()
    rhs_phd = solver._empty_channel_store(solver.phd_grid)
    for iq, Q in enumerate(solver.phd_grid.q_list):
        for s1, s2, s3, s4 in solver.spin_blocks:
            ker = k_ref.compute_ph_kernel(
                gamma, solver.patchsets, Q,
                incoming_spins=(s1, s3),
                outgoing_spins=(s4, s2),
                config=cfg,
                allowed_spin_blocks=solver.allowed_spin_blocks,
            )
            rhs_phd[(s1, s2, s3, s4, iq)] = np.asarray(ker.matrix, dtype=complex)
    timings["phd_time"] = time.perf_counter() - t

    t = time.perf_counter()
    rhs_phc = solver._empty_channel_store(solver.phc_grid)
    for iq, Q in enumerate(solver.phc_grid.q_list):
        for s1, s2, s3, s4 in solver.spin_blocks:
            ker = k_ref.compute_phc_kernel(
                gamma, solver.patchsets, Q,
                incoming_spins=(s1, s2),
                outgoing_spins=(s3, s4),
                config=cfg,
                allowed_spin_blocks=solver.allowed_spin_blocks,
            )
            rhs_phc[(s1, s2, s3, s4, iq)] = np.asarray(ker.matrix, dtype=complex)
    timings["phc_time"] = time.perf_counter() - t

    timings["total_time"] = timings["pp_time"] + timings["phd_time"] + timings["phc_time"]
    return timings, (rhs_pp, rhs_phd, rhs_phc)


def time_rhs_by_channel_opt(solver, T):
    gamma = solver.current_gamma_accessor()
    cfg = solver._flow_config(T)

    timings = {}

    t = time.perf_counter()
    rhs_pp = solver._empty_channel_store(solver.pp_grid)
    for iq, Q in enumerate(solver.pp_grid.q_list):
        for s1, s2, s3, s4 in solver.spin_blocks:
            ker = k_opt.compute_pp_kernel_fast(
                gamma, solver.patchsets, Q,
                incoming_spins=(s1, s2),
                outgoing_spins=(s3, s4),
                config=cfg,
                allowed_spin_blocks=solver.allowed_spin_blocks,
            )
            rhs_pp[(s1, s2, s3, s4, iq)] = np.asarray(ker.matrix, dtype=complex)
    timings["pp_time"] = time.perf_counter() - t

    t = time.perf_counter()
    rhs_phd = solver._empty_channel_store(solver.phd_grid)
    for iq, Q in enumerate(solver.phd_grid.q_list):
        for s1, s2, s3, s4 in solver.spin_blocks:
            ker = k_opt.compute_ph_kernel_fast(
                gamma, solver.patchsets, Q,
                incoming_spins=(s1, s3),
                outgoing_spins=(s4, s2),
                config=cfg,
                allowed_spin_blocks=solver.allowed_spin_blocks,
            )
            rhs_phd[(s1, s2, s3, s4, iq)] = np.asarray(ker.matrix, dtype=complex)
    timings["phd_time"] = time.perf_counter() - t

    t = time.perf_counter()
    rhs_phc = solver._empty_channel_store(solver.phc_grid)
    for iq, Q in enumerate(solver.phc_grid.q_list):
        for s1, s2, s3, s4 in solver.spin_blocks:
            ker = k_opt.compute_phc_kernel_fast(
                gamma, solver.patchsets, Q,
                incoming_spins=(s1, s2),
                outgoing_spins=(s3, s4),
                config=cfg,
                allowed_spin_blocks=solver.allowed_spin_blocks,
            )
            rhs_phc[(s1, s2, s3, s4, iq)] = np.asarray(ker.matrix, dtype=complex)
    timings["phc_time"] = time.perf_counter() - t

    timings["total_time"] = timings["pp_time"] + timings["phd_time"] + timings["phc_time"]
    return timings, (rhs_pp, rhs_phd, rhs_phc)


In [8]:
timing_ref, rhs_ref_manual = time_rhs_by_channel_ref(solver_ref, T0)
timing_opt, rhs_opt_manual = time_rhs_by_channel_opt(solver_opt, T0)

pd.DataFrame([
    {
        "version": "reference",
        **timing_ref,
    },
    {
        "version": "optimized",
        **timing_opt,
    },
    {
        "version": "speedup ref/opt",
        "pp_time": timing_ref["pp_time"] / timing_opt["pp_time"],
        "phd_time": timing_ref["phd_time"] / timing_opt["phd_time"],
        "phc_time": timing_ref["phc_time"] / timing_opt["phc_time"],
        "total_time": timing_ref["total_time"] / timing_opt["total_time"],
    }
])


,version,pp_time,phd_time,phc_time,total_time
0,reference,0.678246,1.118939,1.156594,2.953780
1,optimized,0.159122,0.180998,0.158184,0.498304
2,speedup ref/opt,4.262426,6.182040,7.311716,5.927664


## 5. One manual flow step: separate timing for RHS, norm estimate, apply update

In [9]:
def manual_one_step(solver, T_old, dT, rhs_func_name="compute_channel_rhs"):
    t0 = time.perf_counter()
    rhs_pp, rhs_phd, rhs_phc = getattr(solver, rhs_func_name)(T_old)
    t_rhs = time.perf_counter() - t0

    t0 = time.perf_counter()
    effective_norm = max(solver.state.channel_norm(), solver.bare_vertex_norm, 1e-14)
    rhs_norm = solver._rhs_norm(rhs_pp, rhs_phd, rhs_phc)
    rel_update = abs(dT) * rhs_norm / effective_norm
    n_sub = max(1, int(np.ceil(rel_update / solver.max_relative_update))) if rel_update > solver.max_relative_update else 1
    sub_dT = dT / n_sub
    t_control = time.perf_counter() - t0

    t0 = time.perf_counter()
    for _ in range(n_sub):
        solver._apply_rhs(rhs_pp, rhs_phd, rhs_phc, sub_dT)
    solver.state.T = float(T_old + dT)
    t_apply = time.perf_counter() - t0

    return {
        "T_old": T_old,
        "dT": dT,
        "rhs_norm": rhs_norm,
        "effective_norm": effective_norm,
        "rel_update": rel_update,
        "n_sub": n_sub,
        "channel_norm_after": solver.state.channel_norm(),
        "t_rhs": t_rhs,
        "t_control": t_control,
        "t_apply": t_apply,
        "t_total": t_rhs + t_control + t_apply,
    }

solver_ref_step = f_ref.FRGFlowSolver(
    patchsets=patchsets, bare_gamma=toy_bare_gamma,
    pp_Qs=pp_Qs, ph_Qs=ph_Qs, phc_Qs=phc_Qs,
    T_start=1.0, T_stop=0.7, n_steps=4, nfreq=12, diagnose_every=999,
)
solver_opt_step = f_opt.FRGFlowSolver(
    patchsets=patchsets, bare_gamma=toy_bare_gamma,
    pp_Qs=pp_Qs, ph_Qs=ph_Qs, phc_Qs=phc_Qs,
    T_start=1.0, T_stop=0.7, n_steps=4, nfreq=12, diagnose_every=999,
)

T_old = float(solver_ref_step.temperature_path[0])
dT = float(solver_ref_step.temperature_path[1] - solver_ref_step.temperature_path[0])

rec_ref = manual_one_step(solver_ref_step, T_old, dT)
rec_opt = manual_one_step(solver_opt_step, T_old, dT)

pd.DataFrame([{"version": "reference", **rec_ref}, {"version": "optimized", **rec_opt}])


,version,T_old,dT,rhs_norm,effective_norm,rel_update,n_sub,channel_norm_after,t_rhs,t_control,t_apply,t_total
0,reference,1.0,-0.112096,5.219545,1.27,0.460701,4,0.58509,3.191478,0.000488,0.000392,3.192358
1,optimized,1.0,-0.112096,5.219545,1.27,0.460701,4,0.58509,0.466215,0.000499,0.000375,0.467088


## 6. Manual short flow loop with per-step timing

In [10]:
def manual_flow_loop(solver, n_steps_to_run=3):
    rows = []
    temps = solver.temperature_path
    for i in range(min(n_steps_to_run, len(temps) - 1)):
        T_old = float(temps[i])
        dT = float(temps[i + 1] - temps[i])
        rec = manual_one_step(solver, T_old, dT)
        rec["step"] = i
        rows.append(rec)
    return pd.DataFrame(rows)

solver_ref_loop = f_ref.FRGFlowSolver(
    patchsets=patchsets, bare_gamma=toy_bare_gamma,
    pp_Qs=pp_Qs, ph_Qs=ph_Qs, phc_Qs=phc_Qs,
    T_start=1.0, T_stop=0.7, n_steps=5, nfreq=12, diagnose_every=999,
)
solver_opt_loop = f_opt.FRGFlowSolver(
    patchsets=patchsets, bare_gamma=toy_bare_gamma,
    pp_Qs=pp_Qs, ph_Qs=ph_Qs, phc_Qs=phc_Qs,
    T_start=1.0, T_stop=0.7, n_steps=5, nfreq=12, diagnose_every=999,
)

df_ref_loop = manual_flow_loop(solver_ref_loop, n_steps_to_run=3)
df_opt_loop = manual_flow_loop(solver_opt_loop, n_steps_to_run=3)

print("reference")
display(df_ref_loop)

print("optimized")
display(df_opt_loop)


reference


,T_old,dT,rhs_norm,effective_norm,rel_update,n_sub,channel_norm_after,t_rhs,t_control,t_apply,t_total,step
0,1.000000,-0.085309,5.219545,1.27,0.350609,3,0.445273,2.979928,0.000505,0.000294,2.980728,0
1,0.914691,-0.078031,10.124161,1.27,0.622048,5,1.235213,2.998305,0.000499,0.002550,3.001354,1
2,0.836660,-0.071374,24.288476,1.27,1.365021,10,2.963045,3.934103,0.000484,0.001344,3.935930,2


optimized


,T_old,dT,rhs_norm,effective_norm,rel_update,n_sub,channel_norm_after,t_rhs,t_control,t_apply,t_total,step
0,1.000000,-0.085309,5.219545,1.27,0.350609,3,0.445273,0.459871,0.001137,0.000301,0.461309,0
1,0.914691,-0.078031,10.124161,1.27,0.622048,5,1.235213,0.470962,0.000463,0.000989,0.472413,1
2,0.836660,-0.071374,24.288476,1.27,1.365021,10,2.963045,0.471252,0.000472,0.002073,0.473798,2


## How to interpret the notebook

What should match:
- scalar gamma values,
- small-kernel matrices,
- full RHS stores on the toy model.

What should get faster:
- per-channel timing, especially when the optimized version uses
  - precomputed transfer-index tables in the flow file,
  - matrix contractions in the kernel file.

What this notebook is **not** testing:
- a realistic kagome instability calculation,
- a full heavy production flow,
- diagnosis quality.
